In [ ]:
!pip install transformers datasets scikit-learn accelerate huggingface_hub -q

import os
from pathlib import Path

DATA_DIR  = Path('/kaggle/working/data/nli')
MODEL_DIR = Path('/kaggle/working/models/verify_claim')
RESULTS_DIR = Path('/kaggle/working/results')

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Cartelle create:")
print(f"  {DATA_DIR}")
print(f"  {MODEL_DIR}")
print(f"  {RESULTS_DIR}")

In [ ]:
import json, random
from datasets import load_dataset

LABEL_MAP         = {0: "GROUNDED", 1: "PARTIAL", 2: "NOT_GROUNDED"}
N_TRAIN_PER_CLASS = 10_000
N_VAL             = 5_000
N_TEST            = 5_000
SEED              = 42

def _sample_balanced(split, n_per_class, seed=SEED):
    rng = random.Random(seed)
    buckets = {0: [], 1: [], 2: []}
    for ex in split:
        lbl = ex["label"]
        if lbl in buckets and len(buckets[lbl]) < n_per_class:
            buckets[lbl].append(ex)
        if all(len(v) >= n_per_class for v in buckets.values()):
            break
    out = []
    for lbl, items in buckets.items():
        for ex in items:
            out.append({"claim": ex["hypothesis"], "evidence": ex["premise"],
                        "label": LABEL_MAP[lbl]})
    rng.shuffle(out)
    return out

def _sample_fixed(split, n, seed=SEED):
    rng = random.Random(seed)
    items = list(split)
    rng.shuffle(items)
    return [{"claim": ex["hypothesis"], "evidence": ex["premise"], "label": LABEL_MAP[ex["label"]]}
            for ex in items[:n] if ex["label"] in LABEL_MAP]

def _save_jsonl(examples, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("Scarico MultiNLI da HuggingFace (prima volta ~300MB, poi in cache)...")
dataset = load_dataset("nyu-mll/multi_nli")

train = _sample_balanced(dataset["train"], N_TRAIN_PER_CLASS)
val   = _sample_fixed(dataset["validation_matched"], N_VAL)
test  = _sample_fixed(dataset["validation_mismatched"], N_TEST)

_save_jsonl(train, DATA_DIR / "train.jsonl")
_save_jsonl(val,   DATA_DIR / "val.jsonl")
_save_jsonl(test,  DATA_DIR / "test.jsonl")

for name, split in [("train", train), ("val", val), ("test", test)]:
    counts = {}
    for ex in split:
        counts[ex["label"]] = counts.get(ex["label"], 0) + 1
    print(f"  {name:5s}: {len(split)}  {counts}")

print("\nDataset pronto.")

In [ ]:
import shutil, json
import torch
import numpy as np
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import Dataset

LABEL2ID   = {"GROUNDED": 0, "PARTIAL": 1, "NOT_GROUNDED": 2}
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
BASE_MODEL = "xlm-roberta-base"
MAX_LENGTH = 256

SEARCH_CONFIGS = [
    {"lr": 2e-5, "batch_size": 32, "epochs": 3},
    {"lr": 1e-5, "batch_size": 32, "epochs": 4},
    {"lr": 3e-5, "batch_size": 32, "epochs": 2},
    {"lr": 2e-5, "batch_size": 16, "epochs": 3},
    {"lr": 1e-5, "batch_size": 16, "epochs": 4},
]

print(f"Config search: {len(SEARCH_CONFIGS)} configurazioni")
for i, c in enumerate(SEARCH_CONFIGS):
    print(f"  [{i+1}] lr={c['lr']}  batch={c['batch_size']}  epochs={c['epochs']}")

In [ ]:
class NLIDataset(Dataset):
    def __init__(self, path, tokenizer):
        self.examples = []
        with open(path, encoding='utf-8') as f:
            for line in f:
                ex = json.loads(line)
                if ex['label'] in LABEL2ID:
                    self.examples.append(ex)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(
            ex['claim'], ex['evidence'],
            max_length=MAX_LENGTH, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(LABEL2ID[ex['label']], dtype=torch.long)
        }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':        accuracy_score(labels, preds),
        'macro_f1':        f1_score(labels, preds, average='macro'),
        'f1_grounded':     f1_score(labels, preds, labels=[0], average='macro'),
        'f1_partial':      f1_score(labels, preds, labels=[1], average='macro'),
        'f1_not_grounded': f1_score(labels, preds, labels=[2], average='macro'),
    }


print("Carico tokenizer e dataset...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
train_ds  = NLIDataset(DATA_DIR / 'train.jsonl', tokenizer)
val_ds    = NLIDataset(DATA_DIR / 'val.jsonl',   tokenizer)
test_ds   = NLIDataset(DATA_DIR / 'test.jsonl',  tokenizer)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

In [ ]:
best_f1     = 0.0
best_run    = None
all_results = []

for i, config in enumerate(SEARCH_CONFIGS):
    run_num  = i + 1
    ckpt_dir = MODEL_DIR / f'ckpt_run_{run_num}'
    save_dir = MODEL_DIR / f'model_run_{run_num}'

    print(f"\n{'='*60}")
    print(f"  Run {run_num}/{len(SEARCH_CONFIGS)} — config: {config}")
    print(f"{'='*60}")

    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
    )

    args = TrainingArguments(
        output_dir                  = str(ckpt_dir),
        num_train_epochs            = config['epochs'],
        per_device_train_batch_size = config['batch_size'],
        per_device_eval_batch_size  = 64,
        learning_rate               = config['lr'],
        warmup_steps                = 100,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        save_total_limit            = 1,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_macro_f1',
        greater_is_better           = True,
        logging_steps               = 100,
        report_to                   = 'none',
        fp16                        = torch.cuda.is_available(),
        dataloader_num_workers      = 2,
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    trainer.save_model(str(save_dir))
    val_metrics = trainer.evaluate()

    f1  = val_metrics['eval_macro_f1']
    acc = val_metrics['eval_accuracy']
    print(f"\n  Val macro-F1: {f1:.4f}  |  Accuracy: {acc:.4f}")

    all_results.append({
        "run": run_num, "config": config,
        "val_macro_f1": round(f1, 4), "val_accuracy": round(acc, 4),
        "val_f1_grounded":     round(val_metrics['eval_f1_grounded'], 4),
        "val_f1_partial":      round(val_metrics['eval_f1_partial'], 4),
        "val_f1_not_grounded": round(val_metrics['eval_f1_not_grounded'], 4),
    })

    if ckpt_dir.exists():
        shutil.rmtree(ckpt_dir)
        print("  Checkpoint eliminato — spazio liberato")

    if f1 > best_f1:
        best_f1  = f1
        best_run = run_num
        print(f"  *** Nuovo best: run_{best_run} (macro-F1: {best_f1:.4f}) ***")

    del model, trainer
    torch.cuda.empty_cache()
    print("  Memoria GPU liberata")

In [ ]:
all_results.sort(key=lambda r: r['val_macro_f1'], reverse=True)

print(f"\n{'='*60}")
print(f"  RIEPILOGO CONFIG SEARCH ({len(SEARCH_CONFIGS)} configurazioni)")
print(f"{'='*60}")
print(f"  {'run':<6}  {'lr':<8}  {'batch':<6}  {'epochs':<7}  {'macro-F1':>9}  {'accuracy':>9}")
print(f"  {'─'*56}")
for r in all_results:
    marker = " ← BEST" if r['run'] == best_run else ""
    print(f"  run_{r['run']:<3}  {r['config']['lr']:<8}  {r['config']['batch_size']:<6}  "
          f"{r['config']['epochs']:<7}  {r['val_macro_f1']:>9.4f}  {r['val_accuracy']:>9.4f}{marker}")

best_src = MODEL_DIR / f'model_run_{best_run}'
best_dst = MODEL_DIR / 'best_config_search'
if best_dst.exists():
    shutil.rmtree(best_dst)
shutil.copytree(str(best_src), str(best_dst))
print(f"\n  Modello migliore (run_{best_run}) copiato in {best_dst}")

for r in all_results:
    p = MODEL_DIR / f'model_run_{r["run"]}'
    if p.exists():
        shutil.rmtree(p)

config_results_path = RESULTS_DIR / 'config_search_results.json'
with open(config_results_path, 'w') as f:
    json.dump({"n_configs": len(SEARCH_CONFIGS), "results": all_results}, f, indent=2)
print(f"  Salvato: {config_results_path}")

In [ ]:
device      = 'cuda' if torch.cuda.is_available() else 'cpu'
label_names = [ID2LABEL[i] for i in range(3)]
test_examples = test_ds.examples

def _evaluate(model, examples, device):
    model.eval()
    preds, labels = [], []
    for i, ex in enumerate(examples):
        if i % 500 == 0:
            print(f"  {i}/{len(examples)}...", end="\r")
        enc = tokenizer(ex['claim'], ex['evidence'], max_length=MAX_LENGTH,
                        padding='max_length', truncation=True, return_tensors='pt')
        with torch.no_grad():
            logits = model(
                input_ids=enc['input_ids'].to(device),
                attention_mask=enc['attention_mask'].to(device)
            ).logits
        preds.append(logits.argmax(dim=-1).item())
        labels.append(LABEL2ID[ex['label']])
    print(f"  {len(examples)}/{len(examples)}... done")
    return preds, labels

def _metrics(preds, labels):
    return {
        "accuracy":        round(accuracy_score(labels, preds), 4),
        "macro_f1":        round(f1_score(labels, preds, average='macro'), 4),
        "f1_grounded":     round(f1_score(labels, preds, labels=[0], average='macro'), 4),
        "f1_partial":      round(f1_score(labels, preds, labels=[1], average='macro'), 4),
        "f1_not_grounded": round(f1_score(labels, preds, labels=[2], average='macro'), 4),
    }

# --- Fine-tuned model ---
print(f"Valuto modello fine-tuned (run_{best_run})...")
ft_model = AutoModelForSequenceClassification.from_pretrained(str(best_dst)).to(device)
preds_ft, labels_ft = _evaluate(ft_model, test_examples, device)
ft_metrics = _metrics(preds_ft, labels_ft)
print(f"  Accuracy: {ft_metrics['accuracy']:.4f}  |  Macro-F1: {ft_metrics['macro_f1']:.4f}")
print(classification_report(labels_ft, preds_ft, target_names=label_names, digits=4, zero_division=0))

# --- Baseline: xlm-roberta-base senza fine-tuning ---
print("Valuto baseline (xlm-roberta-base senza fine-tuning)...")
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
).to(device)
preds_base, labels_base = _evaluate(baseline_model, test_examples, device)
base_metrics = _metrics(preds_base, labels_base)
print(f"  Accuracy: {base_metrics['accuracy']:.4f}  |  Macro-F1: {base_metrics['macro_f1']:.4f}")
print(classification_report(labels_base, preds_base, target_names=label_names, digits=4, zero_division=0))

# --- Confronto ---
print(f"\n{'='*60}")
print(f"  CONFRONTO FINALE")
print(f"{'='*60}")
print(f"  {'modello':<45}  {'accuracy':>9}  {'macro-F1':>9}")
print(f"  {'-'*67}")
print(f"  {'xlm-roberta-base (no fine-tuning)':<45}  {base_metrics['accuracy']:>9.4f}  {base_metrics['macro_f1']:>9.4f}")
print(f"  {f'xlm-roberta FT config search (run_{best_run})':<45}  {ft_metrics['accuracy']:>9.4f}  {ft_metrics['macro_f1']:>9.4f}")
print(f"\n  Δ accuracy : {ft_metrics['accuracy'] - base_metrics['accuracy']:+.4f}")
print(f"  Δ macro-F1 : {ft_metrics['macro_f1'] - base_metrics['macro_f1']:+.4f}")

results = {
    "best_run": best_run,
    "best_config": SEARCH_CONFIGS[best_run - 1],
    "baseline": {"model": "xlm-roberta-base (no fine-tuning)", **base_metrics},
    "finetuned": {"model": f"xlm-roberta FT config search (run_{best_run})", **ft_metrics},
    "delta_accuracy": round(ft_metrics['accuracy'] - base_metrics['accuracy'], 4),
    "delta_macro_f1": round(ft_metrics['macro_f1'] - base_metrics['macro_f1'], 4),
}
out_path = RESULTS_DIR / 'baseline_vs_finetuned.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n  Salvato: {out_path}")

In [ ]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO  = "elenamonticchio/verify-claim-xlm-roberta-config-search"

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id=HF_REPO, token=HF_TOKEN, exist_ok=True)
api.upload_folder(folder_path=str(best_dst), repo_id=HF_REPO, token=HF_TOKEN)
print(f"Modello caricato su huggingface.co/{HF_REPO}")